In [ ]:
from pathos.multiprocessing import Process, Queue
import dill
import time

# -------------------- Generic functions (from previous answer) --------------------
def _worker(serialized_func, serialized_args, serialized_kwargs, result_queue=None):
    """
    Global wrapper: deserialize function & arguments, call function,
    and optionally put the result into a queue.
    """
    func = dill.loads(serialized_func)
    args = dill.loads(serialized_args) if serialized_args is not None else ()
    kwargs = dill.loads(serialized_kwargs) if serialized_kwargs is not None else {}
    result = func(*args, **kwargs)
    if result_queue is not None:
        result_queue.put(result)

def create_process(func, args=None, kwargs=None, result_queue=None):
    """
    Create a Process that runs func(*args, **kwargs).
    If result_queue is given, the return value is put into it.
    """
    serialized_func = dill.dumps(func)
    serialized_args = dill.dumps(args) if args is not None else None
    serialized_kwargs = dill.dumps(kwargs) if kwargs is not None else None
    p = Process(target=_worker, args=(serialized_func, serialized_args, serialized_kwargs, result_queue))
    return p

def run_process(process):
    process.start()

def list_processes():
    return active_children()


In [ ]:

def example_function(x, y=10):
    print(f"Running with x={x}, y={y}")
    return x + y

def example_function2(x, y=10, q=None):
    #print(f"Running with x={x}, y={y}")
    time.sleep(10)
    result = x+y
    if q is not None:
        q.put(result)
    return result

if __name__ == '__main__':
    q = Queue()

    #mp.set_start_method('spawn', force=True)
    p = Process(target=example_function2,args=(5, 20, q))
    #p = create_process(example_function2, args=(5, 20), result_queue=q)
    p.start()
    p.join()
    print(q.get(0))

In [ ]:
q = mp.Queue()
q.put(10)
p = mp.Process(target=example_function2,args=(5,), kwargs={"y": 20, "q":q})

In [ ]:
p.start()

In [ ]:
p.join()

In [ ]:
q.get(0)

In [ ]:
adadsas

In [ ]:
# ----------------------------------------------------------------------------------

# Example function that does some work and returns a value
def example_function(x, y=10, delay=0):
    """Simulate work by sleeping, then return x+y."""
    time.sleep(delay)
    result = x + y
    print(f"[Process {mp.current_process().name}] {x} + {y} = {result} (delay={delay})")
    return result


# To avoid recursive process creation on Windows
mp.set_start_method('spawn', force=True)   # optional, ensures spawn behaviour

# -------------------- 1. Run three processes with different arguments ----------
print("=== Running three processes concurrently ===")
queue = mp.Queue()   # Queue to collect results

# Create three processes with different arguments
p1 = create_process(example_function, args=(5,), kwargs={"y": 20, "delay": 1}, result_queue=queue)
p2 = create_process(example_function, args=(10,), kwargs={"y": 30, "delay": 2}, result_queue=queue)
p3 = create_process(example_function, args=(100,), kwargs={"y": 200, "delay": 0.5}, result_queue=queue)

# Start them
run_process(p1)
run_process(p2)
run_process(p3)

# Wait for all to finish
p1.join()
p2.join()
p3.join()

# Collect and print results from the queue (order may not match creation order)
results = []
while not queue.empty():
    results.append(queue.get())
print(f"Results collected: {results}")



# -------------------- 2. Reusing a process concept -----------------------------
print("\n=== Reusing a process for a new calculation ===")
# Create a process for the first calculation
q = mp.Queue()
p_reuse = create_process(example_function, args=(7, 3), result_queue=q)
run_process(p_reuse)
p_reuse.join()
print(f"First calculation result: {q.get()}")

# The same Process object cannot be started again (would raise ValueError)
# Instead, create a *new* Process for the next calculation
q2 = mp.Queue()
p_reuse2 = create_process(example_function, args=(8, 4), result_queue=q2)
run_process(p_reuse2)
p_reuse2.join()
print(f"Second calculation result: {q2.get()}")

# -------------------- 3. Killing a process -------------------------------------
print("\n=== Killing a long-running process ===")
# Create a process that runs for a long time (10 seconds)
q_kill = mp.Queue()
p_kill = create_process(example_function, args=(1, 1), kwargs={"delay": 10}, result_queue=q_kill)
run_process(p_kill)

# Give it a moment to start, then terminate it
time.sleep(2)
print("Terminating the process...")
p_kill.terminate()
p_kill.join()   # Clean up after termination
print("Process terminated. No result will appear in the queue.")

# Verify the queue is empty
print(f"Queue empty after termination: {q_kill.empty()}")

In [ ]:
from pathos.pools import ProcessPool
from multiprocessing import Pool
pool = Pool(4)
def f(x):
    return x[0] +x[1]

In [ ]:
pool.map(f, zip([1, 2, 3, 4], [5, 4, 3, 2]))


In [ ]:
pool.ncpus

In [ ]:
pool.nodes

In [ ]:
import concurrent.futures
import math

PRIMES = [
    112272535095293,
    112582705942171,
    112272535095293,
    115280095190773,
    115797848077099,
    1099726899285419]

def is_prime(n):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False

    sqrt_n = int(math.floor(math.sqrt(n)))
    for i in range(3, sqrt_n + 1, 2):
        if n % i == 0:
            return False
    return True

def main():
    with concurrent.futures.ProcessPoolExecutor() as executor:
        for number, prime in zip(PRIMES, executor.map(is_prime, PRIMES)):
            print('%d is prime: %s' % (number, prime))

if __name__ == '__main__':
    main()

In [1]:
import multiprocessing as mp
import dill
from multi import example_function2, create_process




# Replace the default pickler used by multiprocessing with dill's pickler
#mp.reduction.ForkingPickler = dill.Pickler

if __name__ == '__main__':
    # Force 'spawn' start method (common on Windows, and needed in Jupyter)
    mp.set_start_method('spawn', force=True)

    q = mp.Queue()
    #p = mp.Process(target=example_function2, args=(5, 20, q))
    p = create_process(example_function2, args=(5, 20), result_queue=q)
    p.start()
    p.join()
    print(q.get())   # Output: 25

25


In [2]:
import multiprocessing as mp

In [2]:
p.is_alive()

False

In [3]:
p.start()

AssertionError: cannot start a process twice